# Flowbased
In this file you will find functions that creates flow-parameters and constraints in a scalable way. 


In [1]:
include(dirname(dirname(pwd()))*"\\src\\TuLiPa.jl");
using .TuLiPa
using Dates, DataFrames, CSV, JSON, Plots, JuMP, HiGHS
include(dirname(dirname(pwd())) * raw"\demos\other\flowbased_utils.jl")

remove_cap_for_flowbased_transm (generic function with 1 method)

### Create flows
The data for this part should be replaced with data from JulES

In [2]:
PowerA = 1.0
PowerA_cap = 20.0
DemandA = 10.0
PowerB_cap = 1000000000.0
PowerB = 100000000.0
DemandB = 100.0
PowerC_cap = 1000000000.0 
PowerC = 100.0
DemandC = 10.0


network = Dict(
    "A" => Dict("Demand" => DemandA, "Power" => PowerA, "Power_cap" => PowerA_cap),
    "B" => Dict("Demand" => DemandB, "Power" => PowerB, "Power_cap" => PowerB_cap),
    "C" => Dict("Demand" => DemandC, "Power" => PowerC, "Power_cap" => PowerC_cap)
)

Dict{String, Dict{String, Float64}} with 3 entries:
  "B" => Dict("Power"=>1.0e8, "Power_cap"=>1.0e9, "Demand"=>100.0)
  "A" => Dict("Power"=>1.0, "Power_cap"=>20.0, "Demand"=>10.0)
  "C" => Dict("Power"=>100.0, "Power_cap"=>1.0e9, "Demand"=>10.0)

In [3]:
function make_connection(from, to, prefix = "")  
    transm_from = replace(from, ("PowerBalance_" => ""))
    transm_to = replace(to, ("PowerBalance_" => ""))
    
    flow_name = "Transm_$(transm_from)->$(transm_to)" * prefix
    arrow_name_from = "arrow_from$(from)$(to)" * prefix
    arrow_name_to = "arrow_to$(from)$(to)" * prefix
    balance_from = "$(from)"
    balance_to = "$(to)"
    #cap_name = "$(flow_name)_cap"
    elem = [
        DataElement("Flow", "BaseFlow", flow_name, Dict()),    
        DataElement("Arrow", "BaseArrow", arrow_name_from, Dict("Balance" => String(balance_from), "Flow" => flow_name, "Conversion" => 1.0, "Direction" => "Out")),
        DataElement("Arrow", "BaseArrow", arrow_name_to, Dict("Balance" => String(balance_to), "Flow" => flow_name, "Conversion" => 1.0, "Direction" => "In")),
    ]    
    return elem
end

make_connection (generic function with 2 methods)

In [4]:
for area in keys(network)
    println(area)
    println(typeof(area))
end


B
String
A
String
C
String


In [5]:
function create_elements(network_info, df_flowbased_grid)
        elements = DataElement[]
        for area in keys(network_info)
                new_elements = [
                        DataElement("Balance", "BaseBalance", "$area", Dict{Any, Any}("Commodity" => "Power")),    
                        DataElement("Flow", "BaseFlow", "Power$area", Dict{Any, Any}()),
                        DataElement("Arrow", "BaseArrow", "Arrow$area", Dict{Any, Any}("Balance" => area, "Flow" => "Power$area", "Conversion" => 1.0, "Direction" => "In")),
                        DataElement("Cost", "CostTerm", "Power$(area)_cost", Dict{Any, Any}("Param" => network_info[area]["Power"], "WhichInstance" => "Power$area", "WhichConcept" => "Flow", "Direction" => "In")),
                        DataElement("Capacity", "PositiveCapacity", "Power$(area)_cap", Dict{Any, Any}("Param" => "Power$(area)_cap", "WhichInstance" => "Power$area", "WhichConcept" => "Flow", "Bound" => "Upper")),
                        DataElement("Param", "MWToGWhSeriesParam", "Power$(area)_cap", Dict{Any, Any}("Level" => network_info[area]["Power_cap"], "Profile" => 1.0)),
                        DataElement("Param", "MWToGWhSeriesParam", "Demand$area", Dict("Level" => network_info[area]["Demand"] , "Profile" => 1.0)),
                        DataElement("RHSTerm", "BaseRHSTerm", "Demand$area", Dict{Any, Any}("Balance" => area, "Param" => "Demand$area", "Direction" => "Out")),
                ]
                elements = vcat(elements, new_elements)
        end
        
        # Create flow-parameters for the rows where Border == 1, in both direction
        flow_parameters_1 = [make_connection(df_flowbased_grid.emps_area0[i], df_flowbased_grid.emps_area1[i]) for i in 1:nrow(df_flowbased_grid) if df_flowbased_grid.border[i]]
        flow_parameters_2 = [make_connection(df_flowbased_grid.emps_area1[i], df_flowbased_grid.emps_area0[i]) for i in 1:nrow(df_flowbased_grid) if df_flowbased_grid.border[i]]

        flow_parameters = vcat(flow_parameters_1..., flow_parameters_2...)  # Merge and flatten
        
        elements = vcat(elements, flow_parameters)

        power_horizon = SequentialHorizon(1, Day(1))
        push!(elements, getelement(COMMODITY_CONCEPT, "BaseCommodity", "Power", (HORIZON_CONCEPT, power_horizon)))
        addscenariotimeperiod!(elements, "ScenarioTimePeriod", getisoyearstart(1981), getisoyearstart(1983));
        
        return elements
end

create_elements (generic function with 1 method)

## PTDF-matrix

In [6]:
file = "./assets/3noder_enkeltlinjer_1.csv"
df_flowbased_grid = DataFrame(CSV.File(file))
df_flowbased_grid.Border = df_flowbased_grid.Border .== 1  # Convert to boolean
rename!(df_flowbased_grid, :Border => :border)
#select!(df_ptdf, Not([:emps_area0, :emps_area1]))

println(df_flowbased_grid)

6×8 DataFrame
 Row │ CnecName     emps_area0  emps_area1  A        B         C      RAM        border 
     │ String15     String1     String1     Float64  Float64   Int64  Float64?   Bool   
─────┼──────────────────────────────────────────────────────────────────────────────────
   1 │ BORDER_A->B  A           B           0.33333  -0.33333      0  missing      true
   2 │ BORDER_A->C  A           C           0.66667   0.33333      0  missing      true
   3 │ BORDER_B->C  B           C           0.33333   0.66667      0  missing      true
   4 │ 243          A           B           0.33333  -0.33333      0     2300.0   false
   5 │ 465          A           C           0.6667    0.33333      0     1000.0   false
   6 │ 543          B           C           0.33333   0.66667      0      300.0   false


In [7]:
elements = create_elements(network, df_flowbased_grid)

44-element Vector{DataElement}:
 DataElement("Balance", "BaseBalance", "B", Dict{Any, Any}("Commodity" => "Power"))
 DataElement("Flow", "BaseFlow", "PowerB", Dict{Any, Any}())
 DataElement("Arrow", "BaseArrow", "ArrowB", Dict{Any, Any}("Balance" => "B", "Flow" => "PowerB", "Conversion" => 1.0, "Direction" => "In"))
 DataElement("Cost", "CostTerm", "PowerB_cost", Dict{Any, Any}("Param" => 1.0e8, "WhichInstance" => "PowerB", "WhichConcept" => "Flow", "Direction" => "In"))
 DataElement("Capacity", "PositiveCapacity", "PowerB_cap", Dict{Any, Any}("Param" => "PowerB_cap", "WhichInstance" => "PowerB", "WhichConcept" => "Flow", "Bound" => "Upper"))
 DataElement("Param", "MWToGWhSeriesParam", "PowerB_cap", Dict{Any, Any}("Level" => 1.0e9, "Profile" => 1.0))
 DataElement("Param", "MWToGWhSeriesParam", "DemandB", Dict("Level" => 100.0, "Profile" => 1.0))
 DataElement("RHSTerm", "BaseRHSTerm", "DemandB", Dict{Any, Any}("Balance" => "B", "Param" => "DemandB", "Direction" => "Out"))
 DataElement("

In [8]:
df_flowbased_grid

Row,CnecName,emps_area0,emps_area1,A,B,C,RAM,border
,String15,String1,String1,Float64,Float64,Int64,Float64?,Bool
1,BORDER_A->B,A,B,0.33333,-0.33333,0,missing,true
2,BORDER_A->C,A,C,0.66667,0.33333,0,missing,true
3,BORDER_B->C,B,C,0.33333,0.66667,0,missing,true
4,243,A,B,0.33333,-0.33333,0,2300.0,false
5,465,A,C,0.6667,0.33333,0,1000.0,false
6,543,B,C,0.33333,0.66667,0,300.0,false


In [ ]:
function process_ptdf_matrix(df, remove_nonexisting_emps)

    function _fix_col_names(df)
        df[!, "border"] = occursin.("BORDER", df.CnecName);

        df[!, "CnecName"] .= ifelse.(
        df.border,
        "Transm_" .* df[!, :emps_area0] .* "->" .* df[!, :emps_area1],
        df[!, :CnecName]
        )

        df = select(df, Not([:emps_area0, :emps_area1]))
        return df
    end

    df_new = copy(df)
    #df = correct_line_direction(df)
    df_new= _fix_col_names(df_new)
    
    #= df = select(df, non_value_cols, Not(non_value_cols))
    
    df = round_ptdf_values(df) =#

    if remove_nonexisting_emps
        # removes the lines that are connected in N490-model, but not connected in EMPS.
        is_in(x) = x in ["Transm_NORGEMIDT->OSTLAND", "Transm_MOERE->VESTMIDT", "Transm_OSTLAND->SVER-SE3", "Transm_HALLINGDAL->SORLAND"]
        df_new = df_new[is_in.(df_new.CnecName) .== false, :]
    end

    return df_new
end

process_ptdf_matrix(df_flowbased_grid, true)

Row,CnecName,A,B,C,RAM,border
,Abstract…,Float64,Float64,Int64,Float64?,Bool
1,Transm_A->B,0.33333,-0.33333,0,missing,true
2,Transm_A->C,0.66667,0.33333,0,missing,true
3,Transm_B->C,0.33333,0.66667,0,missing,true
4,243,0.33333,-0.33333,0,2300.0,false
5,465,0.6667,0.33333,0,1000.0,false
6,543,0.33333,0.66667,0,300.0,false


In [ ]:
transm = [e for e in elements if is_transmission(e)]
#df_ptdf = process_ptdf_matrix(df_flowbased_grid)
df_ptdf = process_ptdf_matrix(df_flowbased_grid, false)



Row,CnecName,A,B,C,RAM,border
,Abstract…,Float64,Float64,Int64,Float64?,Bool
1,Transm_A->B,0.33333,-0.33333,0,missing,true
2,Transm_A->C,0.66667,0.33333,0,missing,true
3,Transm_B->C,0.33333,0.66667,0,missing,true
4,243,0.33333,-0.33333,0,2300.0,false
5,465,0.6667,0.33333,0,1000.0,false
6,543,0.33333,0.66667,0,300.0,false


In [11]:
T = DataElement
flow_based = make_flowbased(df_ptdf, transm)
elements = vcat(elements,  flow_based)

50-element Vector{DataElement}:
 DataElement("Balance", "BaseBalance", "B", Dict{Any, Any}("Commodity" => "Power"))
 DataElement("Flow", "BaseFlow", "PowerB", Dict{Any, Any}())
 DataElement("Arrow", "BaseArrow", "ArrowB", Dict{Any, Any}("Balance" => "B", "Flow" => "PowerB", "Conversion" => 1.0, "Direction" => "In"))
 DataElement("Cost", "CostTerm", "PowerB_cost", Dict{Any, Any}("Param" => 1.0e8, "WhichInstance" => "PowerB", "WhichConcept" => "Flow", "Direction" => "In"))
 DataElement("Capacity", "PositiveCapacity", "PowerB_cap", Dict{Any, Any}("Param" => "PowerB_cap", "WhichInstance" => "PowerB", "WhichConcept" => "Flow", "Bound" => "Upper"))
 DataElement("Param", "MWToGWhSeriesParam", "PowerB_cap", Dict{Any, Any}("Level" => 1.0e9, "Profile" => 1.0))
 DataElement("Param", "MWToGWhSeriesParam", "DemandB", Dict("Level" => 100.0, "Profile" => 1.0))
 DataElement("RHSTerm", "BaseRHSTerm", "DemandB", Dict{Any, Any}("Balance" => "B", "Param" => "DemandB", "Direction" => "Out"))
 DataElement("

In [12]:
modelobjects = getmodelobjects(elements)
println(modelobjects)
modelobjects

Dict{Id, Any}(Id("Flow", "Transm_C->B") => BaseFlow(Id("Flow", "Transm_C->B"), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing), nothing, LowerZeroCapacity(), Cost[], nothing, Arrow[BaseArrow(Id("Arrow", "arrow_fromCB"), BaseBalance(Id("Balance", "C"), BaseCommodity(Id("Commodity", "Power"), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing)), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing), RHSTerm[BaseRHSTerm(Id("RHSTerm", "DemandC"), MWToGWhSeriesParam{ConstantTimeVector, ConstantTimeVector}(ConstantTimeVector(10.0), ConstantTimeVector(1.0)), false, Dict{Any, Any}())], Dict{Any, Any}()), BaseConversion(ConstantParam{Float64}(1.0)), nothing, false), BaseArrow(Id("Arrow", "arrow_toCB"), BaseBalance(Id("Balance", "B"), BaseCommodity(Id("Commodity", "Power"), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothing)), SequentialHorizon(SequentialPeriods([(1, Millisecond(86400000))]), nothin

Dict{Id, Any} with 18 entries:
  Id("Flow", "Transm_C->B")      => BaseFlow(Id("Flow", "Transm_C->B"), Sequent…
  Id("FlowBased", "Transm_A->B") => BaseFlowBased{Float64}(Id("FlowBased", "Tra…
  Id("Flow", "Transm_B->C")      => BaseFlow(Id("Flow", "Transm_B->C"), Sequent…
  Id("Balance", "C")             => BaseBalance(Id("Balance", "C"), BaseCommodi…
  Id("FlowBased", "543")         => BaseFlowBased{Float64}(Id("FlowBased", "543…
  Id("Balance", "B")             => BaseBalance(Id("Balance", "B"), BaseCommodi…
  Id("FlowBased", "Transm_A->C") => BaseFlowBased{Float64}(Id("FlowBased", "Tra…
  Id("FlowBased", "243")         => BaseFlowBased{Float64}(Id("FlowBased", "243…
  Id("Flow", "Transm_A->C")      => BaseFlow(Id("Flow", "Transm_A->C"), Sequent…
  Id("Flow", "Transm_C->A")      => BaseFlow(Id("Flow", "Transm_C->A"), Sequent…
  Id("Flow", "Transm_B->A")      => BaseFlow(Id("Flow", "Transm_B->A"), Sequent…
  Id("FlowBased", "465")         => BaseFlowBased{Float64}(Id("FlowBased", "46

In [13]:
mymodel = JuMP.Model(HiGHS.Optimizer)
prob = JuMP_Prob(modelobjects, mymodel)
datatime = getisoyearstart(2023)
scenariotime = getisoyearstart(1981)
prob_time = TwoTime(datatime, scenariotime)
update!(prob, prob_time)

In [14]:

solve!(prob)
print(prob.model)
print(solution_summary(prob.model, verbose = true))

* Solver : HiGHS

* Status
  Result count       : 1
  Termination status : OPTIMAL
  Message from the solver:
  "kHighsModelStatusOptimal"

* Candidate solution (result #1)
  Primal status      : FEASIBLE_POINT
  Dual status        : FEASIBLE_POINT
  Objective value    : 2.40480e+02
  Objective bound    : 0.00000e+00
  Relative gap       : Inf
  Dual objective value : 2.40480e+02
  Primal solution :
    FlowPowerA[1] : 4.80000e-01
    FlowPowerB[1] : 0.00000e+00
    FlowPowerC[1] : 2.40000e+00
    FlowTransm_A->B[1] : 8.79991e-01
    FlowTransm_A->C[1] : 0.00000e+00
    FlowTransm_B->A[1] : 0.00000e+00
    FlowTransm_B->C[1] : 0.00000e+00
    FlowTransm_C->A[1] : 6.39991e-01
    FlowTransm_C->B[1] : 1.52001e+00
  Dual solution :
    BalanceA[1] : -1.00000e+02
    BalanceB[1] : -1.00000e+02
    BalanceC[1] : -1.00000e+02
    FlowBased243in[1] : 0.00000e+00
    FlowBased243out[1] : 0.00000e+00
    FlowBased465in[1] : 0.00000e+00
    FlowBased465out[1] : 0.00000e+00
    FlowBased543in[1] 